# Patch grounding v3 — matched 10-epoch pilot
Upload `kaggle_patch_v3_source.zip` and `grounding_patch_v3_256.h5` as Input; import this notebook and enable GPU. Test is reporting only; checkpoint selection prioritizes valid predictions within 1.5cm over all validation targets, then penalized error (invalid = 1 metre). Wrist/FK fusion is not trained here.


In [ ]:
%pip install -q 'mujoco>=3.2,<4' h5py numpy


In [ ]:
from pathlib import Path
import zipfile, sys, subprocess, torch
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
def unique(name):
    hits=list(INPUT.rglob(name))
    if len(hits)!=1: raise ValueError(f'Expected exactly one {name}, found {hits}')
    return hits[0]
SOURCE=unique('kaggle_patch_v3_source.zip'); DATA=unique('grounding_patch_v3_256.h5')
ROOT=WORK/'omni_patch_v3_source'
if ROOT.exists(): raise FileExistsError('Use a fresh notebook session')
with zipfile.ZipFile(SOURCE) as z:
    for info in z.infolist():
        p=Path(info.filename)
        if p.is_absolute() or '..' in p.parts: raise ValueError('Unsafe archive path')
    z.extractall(ROOT)
assert torch.cuda.is_available(), 'Enable GPU accelerator'
def run(*args): subprocess.run([sys.executable, *map(str,args)],cwd=ROOT,check=True)
EPOCHS=10
OUT=WORK/'grounding_patch_v3_gpu'
run('-m','scripts.train_patch_grounding','--dataset',DATA,'--output',OUT,'--device','cuda','--epochs',EPOCHS)


In [ ]:
for split in ('val','test'):
    run('-m','scripts.eval_patch_grounding','--dataset',DATA,'--checkpoint',OUT/'best.pt','--split',split,'--output',WORK/f'patch_{split}.json')
import shutil
bundle=WORK/'patch_v3_outputs'
bundle.mkdir()
shutil.copytree(OUT,bundle/OUT.name)
for split in ('val','test'): shutil.copy2(WORK/f'patch_{split}.json',bundle)
shutil.make_archive(str(WORK/'patch_v3_outputs'),'zip',bundle)
print('Download /kaggle/working/patch_v3_outputs.zip; evaluate closed-loop on Dell.')
